# Simulación del robot planar de 2 GDL

Notebook basada en el código fuente mostrado en el ejemplo 5.4 del simulador del robot planar de 2 GDL.


## Modelo dinámico

La dinámica se expresa como:

$$\tau = M(q)\ddot q + C(q,\dot q)\dot q + g(q) + f(\dot q)$$

Despejando la aceleración:

$$\ddot q = M(q)^{-1}[\tau - C(q,\dot q)\dot q - g(q) - f(\dot q)]$$


In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom scipy.integrate import solve_ivpdef robot_2gdl(t, x):    q1 = x[0]    q2 = x[1]    qp1 = x[2]    qp2 = x[3]    qp = np.array([qp1, qp2])    # Matriz de inercia    M = np.array([        [3.117 + 0.2*np.cos(q2), 0.108 + 0.1*np.cos(q2)],        [0.108 + 0.1*np.cos(q2), 0.108]    ])    # Matriz de Coriolis y fuerzas centrífugas    C = np.array([        [-0.2*np.sin(q2)*qp2, -0.1*np.sin(q2)*qp2],        [0.1*np.sin(q2)*qp1, 0]    ])    # Vector de gravedad    g = np.array([        39.3*np.sin(q1) + 1.95*np.sin(q1 + q2),        1.95*np.sin(q1 + q2)    ])    # Vector de fricción viscosa y seca    f = np.array([        1.86*qp1 + 1.93*np.sign(qp1),        0.16*qp2 + 0.3*np.sign(qp2)    ])    # Pares aplicados por los motores    tau = np.array([        (1 - np.exp(-0.8*t))*32.0 + 56*np.sin(16*t + 0.1) + 12*np.sin(20*t + 0.15),        (1 - np.exp(-1.8*t))*1.2 + 8*np.sin(26*t + 0.08) + 2*np.sin(12*t + 0.34)    ])    # Aceleración articular    qpp = np.linalg.solve(M, tau - C @ qp - g - f)    return [qp1, qp2, qpp[0], qpp[1]]# Parámetros de simulaciónti = 0h = 0.002tf = 10t_eval = np.arange(ti, tf + h, h)# Condiciones iniciales: [q1, q2, q̇1, q̇2]x0 = [0, 0, 0, 0]# Solución numéricasol = solve_ivp(    robot_2gdl,    [ti, tf],    x0,    t_eval=t_eval,    method='RK45',    rtol=1e-3,    atol=1e-6)# Conversión de radianes a gradosq1_deg = np.rad2deg(sol.y[0])q2_deg = np.rad2deg(sol.y[1])# Gráfica de posiciones articularesplt.figure(figsize=(10, 5))plt.plot(sol.t, q1_deg, label='q1')plt.plot(sol.t, q2_deg, label='q2')plt.xlabel('Tiempo [s]')plt.ylabel('Posición articular [grados]')plt.title('Simulación del robot planar de 2 GDL')plt.grid(True)plt.legend()plt.show()